# OASIS-2 Minimal Colab Training

This notebook is a clean Colab-first path for OASIS-2 training.
It clones the repo, installs dependencies, runs the OASIS-2 training pipeline with live logs, and prints the saved summary.

In [4]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Cerebrasensecloud')
OASIS2_BUNDLE_ROOT = DRIVE_ROOT / 'OASIS-2'
RUNTIME_ROOT = DRIVE_ROOT / 'backend_runtime'
RUN_NAME = 'oasis2_colab_baseline_v2'

for name, path in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'OASIS2_BUNDLE_ROOT': OASIS2_BUNDLE_ROOT,
    'RUNTIME_ROOT': RUNTIME_ROOT,
}.items():
    print(name, path.exists(), path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT True /content/drive/MyDrive/Cerebrasensecloud
OASIS2_BUNDLE_ROOT True /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
RUNTIME_ROOT True /content/drive/MyDrive/Cerebrasensecloud/backend_runtime


In [5]:
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path('/content/cerebrasense')
BACKEND_ROOT = REPO_ROOT / 'alz_backend'
REPO_URL = 'https://github.com/Billrichard209/cerebrasense.git'
REQUIRED_COMMIT = 'e577752'

def run_checked(cmd, *, cwd=None, label=None):
    print(f"RUNNING {label or cmd[0]}: {' '.join(cmd)}", flush=True)
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, flush=True)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed ({label or cmd[0]}): {' '.join(cmd)}")
    return completed

for stale_root in [Path('/content/cerebrasense'), Path('/content/Cerebrasense-')]:
    if stale_root.exists():
        shutil.rmtree(stale_root)

run_checked(['git', 'clone', REPO_URL, str(REPO_ROOT)], cwd='/content', label='git-clone')
run_checked(['git', 'checkout', 'main'], cwd=REPO_ROOT, label='git-checkout-main')
run_checked(['python3', '-m', 'pip', 'install', '-r', str(BACKEND_ROOT / 'requirements-colab.txt')], cwd=REPO_ROOT, label='pip-install')

repo_commit = run_checked(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, label='git-rev-parse').stdout.strip()
print('repo_commit =', repo_commit)
print('required_commit =', REQUIRED_COMMIT)




RUNNING git-clone: git clone https://github.com/Billrichard209/cerebrasense.git /content/cerebrasense
Cloning into '/content/cerebrasense'...

RUNNING git-checkout-main: git checkout main
Your branch is up to date with 'origin/main'.

Already on 'main'

RUNNING pip-install: python3 -m pip install -r /content/cerebrasense/alz_backend/requirements-colab.txt
  Using cached simpleitk-2.5.4-cp311-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (7.4 kB)
  Using cached monai-1.5.2-py3-none-any.whl.metadata (13 kB)
Using cached simpleitk-2.5.4-cp311-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (52.8 MB)
Using cached monai-1.5.2-py3-none-any.whl (2.7 MB)

RUNNING git-rev-parse: git rev-parse HEAD
890becab980b829ee59e3a1882f1cc4845c0bd2f

repo_commit = 890becab980b829ee59e3a1882f1cc4845c0bd2f
required_commit = 8b847ca


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    'scripts/train_oasis2_colab.py',
    '--project-root', str(BACKEND_ROOT),
    '--runtime-root', str(RUNTIME_ROOT),
    '--bundle-root', str(OASIS2_BUNDLE_ROOT),
    '--run-name', RUN_NAME,
    '--epochs', '20',
    '--batch-size', '1',
    '--gradient-accumulation-steps', '1',
    '--num-workers', '0',
    '--image-size', '64', '64', '64',
    '--seed', '42',
    '--split-seed', '42',
    '--device', 'auto',
]

print('RUNNING:', ' '.join(cmd), flush=True)
process = subprocess.Popen(
    cmd,
    cwd=BACKEND_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'OASIS-2 training runner failed with exit code {return_code}')

print('OASIS-2 training runner completed successfully.', flush=True)


RUNNING: /usr/bin/python3 scripts/train_oasis2_colab.py --project-root /content/cerebrasense/alz_backend --runtime-root /content/drive/MyDrive/Cerebrasensecloud/backend_runtime --bundle-root /content/drive/MyDrive/Cerebrasensecloud/OASIS-2 --run-name oasis2_colab_baseline_v2 --epochs 20 --batch-size 1 --gradient-accumulation-steps 1 --num-workers 0 --image-size 64 64 64 --seed 42 --split-seed 42 --device auto
Starting OASIS-2 Colab bundle pipeline
Resolved OASIS-2 bundle root: /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
Runtime data root: /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/data
Runtime outputs root: /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs
Inspecting uploaded OASIS-2 bundle
Upload bundle inspection status: pass
Resolving OASIS-2 metadata template source
Resolved metadata template source: /content/drive/MyDrive/Cerebrasensecloud/OASIS-2/backend_reference/oasis2_metadata_template.csv
Running preflight runtime refresh from Drive bundle
Re

In [ ]:
from pathlib import Path
import json
import pandas as pd

blocked_summary_path = RUNTIME_ROOT / 'outputs' / 'reports' / 'onboarding' / 'oasis2_colab_bundle_summary.json'
trained_summary_path = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME / 'reports' / 'colab_run_summary.json'

print('blocked_summary_path =', blocked_summary_path)
print('trained_summary_path =', trained_summary_path)

summary_candidates = [trained_summary_path, blocked_summary_path]
existing_summaries = [path for path in summary_candidates if path.exists()]
if not existing_summaries:
    raise FileNotFoundError('Expected an OASIS-2 summary JSON, but none was found in the runtime outputs.')

summary_path = max(existing_summaries, key=lambda path: path.stat().st_mtime)
print('using_summary_path =', summary_path)

summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))
print('training_ready =', summary.get('training_ready'))
print('training_started =', summary.get('training_started'))
print('blocked_reason =', summary.get('blocked_reason'))
print('run_root =', summary.get('run_root'))
print('best_checkpoint =', summary.get('best_checkpoint'))

metrics_path = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME / 'metrics' / 'epoch_metrics.csv'
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    print(metrics.tail())


In [ ]:
from pathlib import Path
import pandas as pd

run_root = Path('/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_baseline_v2')
print('run_root:', run_root.exists(), run_root)

metrics = run_root / 'metrics' / 'epoch_metrics.csv'
best_ckpt = run_root / 'checkpoints' / 'best_model.pt'
last_ckpt = run_root / 'checkpoints' / 'last_model.pt'

for p in [metrics, best_ckpt, last_ckpt]:
    print(p.name, p.exists())

if metrics.exists():
    df = pd.read_csv(metrics)
    print(df.tail())
